# Festa em Cena — análise exploratória de pedidos de Halloween

Projeto do Módulo 1 (SCTEC). Objetivo: ler, limpar e explorar `halloween_vendas.csv`
antes de comparar canais e períodos de venda.

In [1]:
#Bibliotecas usadas no projeto

import re                                               # Expressões regulares (limpeza de texto)
from datetime import datetime                           # Conversão de datas
from pathlib import Path                                # Caminho de arquivos
import pandas as pd                                     # Tabelas (DataFrames)

In [2]:
def ler_csv(caminho):
    """Lê o CSV e devolve um DataFrame com todas as colunas como texto.

    Lemos tudo como texto (dtype=str) para não deixar o Pandas
    adivinhar tipos: assim conseguimos ver valores problemáticos
    como eles vieram e convertê-los depois, com regras.
    """
    arquivo = Path(caminho)
    print("Pasta de execução:", Path.cwd())
    print("Arquivo encontrado:", arquivo.exists())
    df = pd.read_csv(arquivo, sep=",", encoding="utf-8", dtype=str)
    return df

In [3]:
CAMINHO_CSV = "halloween_vendas.csv" #Chama a função e guarda o que ela devolveu na variável

df_original = ler_csv(CAMINHO_CSV) #Usamos o nome “original” porque nunca vamos alterar essa tabela: o PDF exige preservar o original.

print("Total de linhas e colunas:", df_original.shape) # mostra (linhas, colunas). Antes de executar, tente prever: o CSV tem 1.501 linhas de texto, contando o cabeçalho. Quantas linhas o Pandas vai mostrar?
df_original.head(3) # Mostra os 3 primeiros registros, como o Sprint 1 pede.

Pasta de execução: c:\Users\N-lenovo\Documents\SCTEC\festa-em-cena
Arquivo encontrado: True
Total de linhas e colunas: (1500, 14)


,pedido_id,data_venda,produto,categoria,canal_venda,cidade,quantidade,preco_unitario,custo_unitario,desconto_pct,status_pedido,publico,tamanho,data_entrega
0,844,2025-11-21 18:02:35,Fantasia de vampiro,Fantasia,Loja física,Curitiba,3,132.5,72.08,15,Cancelado,Adulto,G,NaN
1,378,2024-10-29 18:41:26,Fantasia de bruxa,FANTASIA!!!,Loja física,Maringá,3,110,57,5,Concluído,Adulto,M,2024-11-03 18:41:26
2,953,2025-09-05 21:16:09,Fantasia de bruxa,Fantasia,Web,Maringá,2,116.6,60.42,5,Concluído,Adulto,M,2025-09-10 21:16:09


Para cada coluna, isso mostra o nome, quantos valores estão preenchidos (non-null) e o tipo. Olhar com atenção para as colunas que têm menos preenchidos que o total de linhas: esses serão os nossos primeiros suspeitos de dados ausentes.

In [4]:
df_original.info()

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   pedido_id       1500 non-null   str  
 1   data_venda      1500 non-null   str  
 2   produto         1500 non-null   str  
 3   categoria       1482 non-null   str  
 4   canal_venda     1500 non-null   str  
 5   cidade          1500 non-null   str  
 6   quantidade      1500 non-null   str  
 7   preco_unitario  1488 non-null   str  
 8   custo_unitario  1493 non-null   str  
 9   desconto_pct    1500 non-null   str  
 10  status_pedido   1500 non-null   str  
 11  publico         1500 non-null   str  
 12  tamanho         1500 non-null   str  
 13  data_entrega    1357 non-null   str  
dtypes: str(14)
memory usage: 164.2 KB


In [5]:
# Conta quantos valores vazios existem em cada coluna
ausentes = df_original.isna().sum()
print(ausentes)

pedido_id           0
data_venda          0
produto             0
categoria          18
canal_venda         0
cidade              0
quantidade          0
preco_unitario     12
custo_unitario      7
desconto_pct        0
status_pedido       0
publico             0
tamanho             0
data_entrega      143
dtype: int64


## Primeira inspeção

- A base tem 1.500 linhas e 14 colunas.
- Colunas com valores vazios: `categoria` (18), `preco_unitario` (12),
  `custo_unitario` (7) e `data_entrega` (143).
- O `info()` só detecta células vazias; valores preenchidos mas inválidos
  (ex.: quantidade 0 ou data escrita errada) serão investigados nos próximos passos.

- Cada linha representa um pedido de um único produto. A coluna `quantidade`
  informa quantas unidades desse produto foram pedidas: uma linha com
  quantidade 3 conta como 1 pedido e 3 unidades.

In [6]:
# Como os textos vieram no arquivo, antes de qualquer limpeza
print(df_original["categoria"].value_counts(dropna=False))
print()
print(df_original["canal_venda"].value_counts(dropna=False))

categoria
Fantasia            892
Acessorio           459
 Fantasia            48
  FANTASIA!!!        39
 Acessorio           28
NaN                  18
  ACESSORIO!!!       16
Name: count, dtype: int64

canal_venda
Web              646
Loja física      535
Mercado          274
 web              20
 loja física      16
 mercado           9
Name: count, dtype: int64


## Sprint 2 — padronização de textos

In [7]:
def limpar_categoria(valor):
    """Limpa um valor da coluna categoria e devolve o texto padronizado.

    Regras:
    1. Vazio (NaN) vira "sem categoria" (exigência do projeto).
    2. strip() tira espaços das pontas; lower() deixa tudo minúsculo.
    3. re.sub() apaga pontuação, mantendo apenas letras e espaços
       (as categorias deste CSV não usam acentos).
    4. Espaços repetidos no meio viram um só.
    5. Se sobrar texto vazio (ex.: "!!!"), também vira "sem categoria".
    6. A primeira letra volta a ser maiúscula, só para apresentação
       (ex.: "fantasia" -> "Fantasia").
    """
    if pd.isna(valor):                              # regra 1
        return "sem categoria"

    texto = valor.strip().lower()                   # regra 2
    texto = re.sub(r"[^a-z\s]", "", texto)          # regra 3
    texto = re.sub(r"\s+", " ", texto).strip()      # regra 4

    if texto == "":                                 # regra 5
        return "sem categoria"
    else:
        return texto.capitalize()                   # regra 6

In [8]:
# Testa a função com exemplos tirados do value_counts
exemplos = ["Fantasia", " Fantasia", "  FANTASIA!!!", "  ACESSORIO!!!", None, "!!!"]

for exemplo in exemplos:
    print(repr(exemplo), "->", repr(limpar_categoria(exemplo)))

'Fantasia' -> 'Fantasia'
' Fantasia' -> 'Fantasia'
'  FANTASIA!!!' -> 'Fantasia'
'  ACESSORIO!!!' -> 'Acessorio'
None -> 'sem categoria'
'!!!' -> 'sem categoria'


In [9]:
# Cópia de trabalho: o df_original nunca é alterado
df_org = df_original.copy()

# Guarda o texto original para mostrar "antes e depois"
df_org["categoria_original"] = df_org["categoria"]

# Aplica a função em cada linha da coluna
df_org["categoria"] = df_org["categoria"].apply(limpar_categoria)

print(df_org["categoria"].value_counts(dropna=False))

categoria
Fantasia         979
Acessorio        503
sem categoria     18
Name: count, dtype: int64


In [10]:
# 1) Vazias que foram preenchidas com "sem categoria"
vazias_preenchidas = df_org["categoria"] == "sem categoria"

# 2) Preenchidas no original, mas que mudaram com a limpeza
nao_vazias_alteradas = (
    df_org["categoria_original"].notna()
    & (df_org["categoria_original"] != df_org["categoria"])
)

print("Categorias vazias preenchidas:", vazias_preenchidas.sum())
print("Categorias não vazias alteradas:", nao_vazias_alteradas.sum())

# Exemplos de antes e depois (um de cada tipo)
print(df_org.loc[nao_vazias_alteradas, ["categoria_original", "categoria"]].drop_duplicates())

Categorias vazias preenchidas: 18
Categorias não vazias alteradas: 131
   categoria_original  categoria
1       FANTASIA!!!     Fantasia
12          Fantasia    Fantasia
18     ACESSORIO!!!    Acessorio
22         Acessorio   Acessorio


In [11]:
# Tradução: texto limpo (minúsculo, sem espaços) -> nome oficial do canal
MAPA_CANAIS = {
    "web": "Web",
    "loja física": "Loja física",
    "mercado": "Mercado",
}

def padronizar_canal(valor):
    """Devolve o nome oficial do canal, ou None se o canal for desconhecido.

    Um canal desconhecido NÃO é adivinhado: vira vazio para ser investigado.
    """
    if pd.isna(valor):
        return None
    chave = valor.strip().lower()
    if chave in MAPA_CANAIS:
        return MAPA_CANAIS[chave]
    else:
        return None

df_org["canal_original"] = df_org["canal_venda"]
df_org["canal_venda"] = df_org["canal_venda"].apply(padronizar_canal)

canais_alterados = df_org["canal_original"] != df_org["canal_venda"]
print(df_org["canal_venda"].value_counts(dropna=False))
print("Canais alterados:", canais_alterados.sum())
print("Canais desconhecidos:", df_org["canal_venda"].isna().sum())

canal_venda
Web            666
Loja física    551
Mercado        283
Name: count, dtype: int64
Canais alterados: 45
Canais desconhecidos: 0


### Resultado da padronização

- **Categoria:** apliquei `strip()`, `lower()` e `re.sub(r"[^a-z\s]", "", texto)`
  para retirar espaços, maiúsculas e pontuação. Vazios viraram "sem categoria".
  - Vazias preenchidas: 18
  - Não vazias alteradas: 131 (ex.: "  FANTASIA!!!" → "Fantasia")
  - Resultado: Fantasia 979, Acessorio 503, sem categoria 18 (total 1.500)
- **Canal:** `strip()` + `lower()` + dicionário de tradução para os nomes oficiais.
  - Canais alterados: 45 (ex.: " web" → "Web"). Canais desconhecidos: 0.
- Os textos originais ficaram nas colunas `categoria_original` e `canal_original`.

## Sprint 3 — validação de quantidade e preço

In [12]:
# Todos os valores diferentes de quantidade, como vieram no arquivo
print(df_org["quantidade"].value_counts(dropna=False))

quantidade
2     462
3     429
1     416
6      54
7      46
4      44
5      42
-1      7
Name: count, dtype: int64


In [13]:
def eh_numero(texto):
    """Devolve True se o texto puder ser convertido em número."""
    try:
        float(texto)          # tenta converter
        return True           # se chegou aqui, deu certo
    except ValueError:
        return False          # se deu erro, não é número

# Teste rápido
for exemplo in ["12.50", "-3", "abc", "12,50"]:
    print(repr(exemplo), "->", eh_numero(exemplo))

'12.50' -> True
'-3' -> True
'abc' -> False
'12,50' -> False


In [14]:
precos = df_org["preco_unitario"]

# Preenchidos, mas que não viram número
nao_numericos = precos.notna() & ~precos.apply(eh_numero)
print("Preços vazios:", precos.isna().sum())
print("Preços preenchidos que não são número:", nao_numericos.sum())
print(precos[nao_numericos].value_counts())

# Entre os que são número: quantos são negativos?
numericos = precos[precos.notna() & ~nao_numericos].astype(float)
print("Preços negativos:", (numericos < 0).sum())
print("Menor preço:", numericos.min(), "| Maior preço:", numericos.max())

Preços vazios: 12
Preços preenchidos que não são número: 0
Series([], Name: count, dtype: int64)
Preços negativos: 0
Menor preço: 22.0 | Maior preço: 132.5


In [15]:
def converter_quantidade(texto):
    """Converte quantidade para inteiro. Devolve None se for inválida.

    Regra do projeto: quantidade deve ser um número inteiro maior que zero.
    Valores inválidos (vazio, texto, decimal, zero ou negativo) não são
    corrigidos nem inventados: ficam vazios e o pedido é marcado com problema.
    """
    if pd.isna(texto):
        return None
    try:
        numero = float(texto)
    except ValueError:
        return None                      # não é número: continua sem parar o programa
    if numero.is_integer() and numero > 0:
        return int(numero)
    else:
        return None


def converter_preco(texto):
    """Converte preço para número decimal. Devolve None se for inválido.

    Regra do projeto: preço deve estar preenchido e não pode ser negativo.
    Preço vazio NÃO vira zero: vazio significa "desconhecido", e zero seria
    um valor inventado.
    """
    if pd.isna(texto):
        return None
    try:
        numero = float(texto)
    except ValueError:
        return None
    if numero >= 0:
        return numero
    else:
        return None

In [16]:
# Colunas convertidas (o texto original continua em "quantidade" e "preco_unitario")
df_org["quantidade_num"] = df_org["quantidade"].apply(converter_quantidade).astype("Int64")
df_org["preco_num"] = df_org["preco_unitario"].apply(converter_preco)

# Marcas (flags) de problema: True = o pedido tem problema nesse campo
df_org["problema_quantidade"] = df_org["quantidade_num"].isna()
df_org["problema_preco"] = df_org["preco_num"].isna()

# O pedido tem PELO MENOS UM problema numérico?
df_org["problema_numerico"] = df_org["problema_quantidade"] | df_org["problema_preco"]

In [17]:
# Contagem por campo, usando um dicionário e um laço
problemas_por_campo = {
    "quantidade": df_org["problema_quantidade"].sum(),
    "preco_unitario": df_org["problema_preco"].sum(),
}
for campo, total in problemas_por_campo.items():
    print(f"Problemas em {campo}: {total}")

# Contagem por pedido (sem somar sobreposições)
ambos = df_org["problema_quantidade"] & df_org["problema_preco"]
print(f"Pedidos com problema nos dois campos: {ambos.sum()}")
print(f"Total de pedidos afetados: {df_org['problema_numerico'].sum()}")

# Exemplos dos registros com problema
print(df_org.loc[df_org["problema_numerico"], ["pedido_id", "quantidade", "preco_unitario"]].head())

Problemas em quantidade: 7
Problemas em preco_unitario: 12
Pedidos com problema nos dois campos: 0
Total de pedidos afetados: 19
    pedido_id quantidade preco_unitario
23       1500          3            NaN
164       630         -1             22
199       625          2            NaN
216      1000          1            NaN
239      1250          1            NaN


In [18]:
# Contagem por campo, usando um dicionário e um laço
problemas_por_campo = {
    "quantidade": df_org["problema_quantidade"].sum(),
    "preco_unitario": df_org["problema_preco"].sum(),
}
for campo, total in problemas_por_campo.items():
    print(f"Problemas em {campo}: {total}")

# Contagem por pedido (sem somar sobreposições)
ambos = df_org["problema_quantidade"] & df_org["problema_preco"]
print(f"Pedidos com problema nos dois campos: {ambos.sum()}")
print(f"Total de pedidos afetados: {df_org['problema_numerico'].sum()}")

# Exemplos dos registros com problema
print(df_org.loc[df_org["problema_numerico"], ["pedido_id", "quantidade", "preco_unitario"]].head())

Problemas em quantidade: 7
Problemas em preco_unitario: 12
Pedidos com problema nos dois campos: 0
Total de pedidos afetados: 19
    pedido_id quantidade preco_unitario
23       1500          3            NaN
164       630         -1             22
199       625          2            NaN
216      1000          1            NaN
239      1250          1            NaN


### Resultado da validação

- **Regra da quantidade:** inteiro maior que zero. 7 pedidos têm quantidade `-1`.
- **Regra do preço:** preenchido e não negativo. 12 pedidos têm preço vazio;
  nenhum é negativo ou não numérico.
- Nenhum pedido tem os dois problemas, por isso o total de pedidos afetados é 19.
- Os registros **não foram apagados nem corrigidos**: ficaram marcados nas colunas
  `problema_quantidade`, `problema_preco` e `problema_numerico`.
- Preço vazio não virou zero, porque zero seria um valor inventado.
- Usei `try/except` para que um valor impossível de converter não interrompa o programa.

## Sprint 4 — datas e regras de negócio

In [19]:
FORMATO_DATA = "%Y-%m-%d %H:%M:%S"   # formato das datas no CSV

def converter_data(texto):
    """Converte o texto de data_venda em datetime, usando datetime.strptime().

    Se o texto estiver vazio ou fora do formato, devolve None (vazio).
    A data NÃO é substituída pela data de hoje nem por qualquer outra:
    o registro é apenas marcado como data inválida.
    """
    if pd.isna(texto):
        return None
    try:
        return datetime.strptime(texto.strip(), FORMATO_DATA)
    except ValueError:
        return None

# Teste com exemplos
exemplos = ["2025-10-12 14:35:00", "2025-13-45 10:00:00", "12/10/2025", "data_invalida", None]
for exemplo in exemplos:
    print(repr(exemplo), "->", converter_data(exemplo))

'2025-10-12 14:35:00' -> 2025-10-12 14:35:00
'2025-13-45 10:00:00' -> None
'12/10/2025' -> None
'data_invalida' -> None
None -> None


In [20]:
# Preserva o texto original (exigência do Projeto)
df_org["data_venda_original"] = df_org["data_venda"]

# Converte; o que falhar vira vazio (NaT = "Not a Time", o vazio das datas)
df_org["data_venda_dt"] = df_org["data_venda"].apply(converter_data)
df_org["data_invalida"] = df_org["data_venda_dt"].isna()

print("Datas válidas:", (~df_org["data_invalida"]).sum())
print("Datas inválidas ou vazias:", df_org["data_invalida"].sum())
print(df_org.loc[df_org["data_invalida"], ["pedido_id", "data_venda_original"]])

Datas válidas: 1496
Datas inválidas ou vazias: 4
    pedido_id data_venda_original
152       740       data_invalida
255       370       data_invalida
714      1480       data_invalida
884      1110       data_invalida


In [21]:
# Partes da data (Int64 aceita vazio nas datas inválidas)
df_org["ano"] = df_org["data_venda_dt"].dt.year.astype("Int64")
df_org["mes"] = df_org["data_venda_dt"].dt.month.astype("Int64")
df_org["dia"] = df_org["data_venda_dt"].dt.day.astype("Int64")
df_org["hora"] = df_org["data_venda_dt"].dt.hour.astype("Int64")


def formatar_data_br(data):
    """Apresenta a data no padrão brasileiro (dd/mm/aaaa) usando strftime()."""
    if pd.isna(data):
        return None
    return data.strftime("%d/%m/%Y")


df_org["data_venda_br"] = df_org["data_venda_dt"].apply(formatar_data_br)

print(df_org[["data_venda_original", "data_venda_br", "ano", "mes", "dia", "hora"]].head())

   data_venda_original data_venda_br   ano  mes  dia  hora
0  2025-11-21 18:02:35    21/11/2025  2025   11   21    18
1  2024-10-29 18:41:26    29/10/2024  2024   10   29    18
2  2025-09-05 21:16:09    05/09/2025  2025    9    5    21
3  2024-11-08 11:29:44    08/11/2024  2024   11    8    11
4  2025-08-23 11:22:05    23/08/2025  2025    8   23    11


A lógica de uma hipótese com “todos”

Para provar “todo pedido sem entrega foi cancelado”, não basta achar muitos cancelados. Basta um único pedido sem entrega com status “Concluído”, por exemplo, para a frase ser falsa. Esse caso que derruba a regra se chama contraexemplo.

Faz sentido como hipótese de negócio: um pedido cancelado nunca é entregue, então não teria data de entrega. Mas será que só os cancelados estão sem data?
Antes de comparar textos, preciso saber como eles estão escritos

In [22]:
print(df_org["status_pedido"].value_counts(dropna=False))

status_pedido
Concluído    1345
Cancelado     109
Devolvido      46
Name: count, dtype: int64


In [23]:
def data_entrega_vazia(valor):
    """True se a data de entrega estiver vazia (NaN ou só espaços)."""
    if pd.isna(valor):
        return True
    elif valor.strip() == "":
        return True
    else:
        return False


# Contadores começam em zero
sem_entrega_cancelados = 0
sem_entrega_outros_status = 0
exemplos_contraexemplo = []          # lista vazia, será preenchida no laço

# Percorre a tabela linha por linha
for indice, linha in df_org.iterrows():
    if data_entrega_vazia(linha["data_entrega"]):
        if linha["status_pedido"] == "Cancelado":
            sem_entrega_cancelados += 1
        else:
            sem_entrega_outros_status += 1
            if len(exemplos_contraexemplo) < 3:
                exemplos_contraexemplo.append(linha["pedido_id"])

print("Sem entrega e cancelados:", sem_entrega_cancelados)
print("Sem entrega com outros status:", sem_entrega_outros_status)

if sem_entrega_outros_status > 0:
    print("Conclusão: hipótese FALSA (há contraexemplos).")
else:
    print("Conclusão: hipótese VERDADEIRA nesta base.")

Sem entrega e cancelados: 109
Sem entrega com outros status: 34
Conclusão: hipótese FALSA (há contraexemplos).


In [24]:
# Flag: o pedido está sem data de entrega? (status e datas NÃO são alterados)
df_org["sem_entrega"] = df_org["data_entrega"].apply(data_entrega_vazia)

# Até 3 contraexemplos
colunas = ["pedido_id", "status_pedido", "data_venda_br", "data_entrega"]
print(df_org.loc[df_org["pedido_id"].isin(exemplos_contraexemplo), colunas])

    pedido_id status_pedido data_venda_br data_entrega
49       1170     Concluído    19/08/2025          NaN
117       990     Concluído    13/10/2025          NaN
152       740     Concluído           NaN          NaN


Conclusão: a hipótese é falsa. Existem 34 pedidos sem data de entrega que não foram cancelados. Os exemplos mostram status Concluído, ou seja, a venda foi registrada como finalizada, mas não há data de entrega.
Todo cancelado está sem entrega. Essa direção é verdadeira, mas nem todo pedido sem entrega está cancelado. Essa é a direção que o projeto perguntou, e ela é falsa.

In [25]:
sem_entrega_nao_cancelado = df_org["sem_entrega"] & (df_org["status_pedido"] != "Cancelado")
print(df_org.loc[sem_entrega_nao_cancelado, "status_pedido"].value_counts())

status_pedido
Concluído    34
Name: count, dtype: int64


In [26]:
# Base: só canais reconhecidos (todos os status entram)
base_canal = df_org[df_org["canal_venda"].notna()]
excluidos_canal = df_org["canal_venda"].isna().sum()

pedidos_por_canal = base_canal.groupby("canal_venda").size().sort_values(ascending=False)
percentual_canal = (100 * pedidos_por_canal / pedidos_por_canal.sum()).round(1)

print(pedidos_por_canal)
print()
print("Participação (%):")
print(percentual_canal)
print("Excluídos por canal desconhecido:", excluidos_canal)

canal_venda
Web            666
Loja física    551
Mercado        283
dtype: int64

Participação (%):
canal_venda
Web            44.4
Loja física    36.7
Mercado        18.9
dtype: float64
Excluídos por canal desconhecido: 0


In [27]:
# Base: só datas válidas (todos os status entram)
base_tempo = df_org[~df_org["data_invalida"]].copy()
excluidos_data = df_org["data_invalida"].sum()

# strftime de novo: transforma a data em texto "2025-10" (ano e mês juntos)
base_tempo["ano_mes"] = base_tempo["data_venda_dt"].dt.strftime("%Y-%m")

pedidos_por_ano_mes = base_tempo.groupby("ano_mes").size().sort_index()
print(pedidos_por_ano_mes)
print("Excluídos por data inválida:", excluidos_data)

ano_mes
2024-08     77
2024-09    104
2024-10    393
2024-11    174
2025-08     62
2025-09     94
2025-10    410
2025-11    182
dtype: int64
Excluídos por data inválida: 4


### Conclusões do Sprint 4
Hipótese da data de entrega. A hipótese de que todo pedido sem data de entrega foi cancelado é falsa. Dos 143 pedidos sem data de entrega, 109 estão cancelados, mas 34 têm status Concluído. Esses 34 são contraexemplos suficientes para derrubar a hipótese. Por outro lado, todos os 109 pedidos cancelados estão sem data de entrega, ou seja, a relação vale em um sentido (“todo cancelado não tem entrega”), mas não no outro. Interpretação: um pedido Concluído sem data de entrega pode ser uma falha de registro ou uma retirada na loja. Limitação: o dado não informa qual dessas explicações é a correta, nem se há outra. Por isso, o status e as datas não foram alterados, apenas marcados na coluna sem_entrega.

Descoberta por canal. A Web concentra o maior número de pedidos registrados: 666 (44,4%), seguida pela Loja física, com 551 (36,7%), e pelo Mercado, com 283 (18,9%). Nenhum pedido ficou de fora por canal desconhecido. Interpretação: o site é o principal canal de entrada de pedidos da Festa em Cena, o que sugere priorizar a experiência de compra e o estoque disponível para a Web na temporada. Limitação: a contagem é de pedidos, e não de unidades nem de receita. Como um pedido pode ter várias unidades, o Mercado pode ter menos pedidos e ainda assim vender muitas fantasias por pedido. Além disso, a contagem inclui pedidos cancelados e a base é fictícia.

Descoberta por período. Outubro concentra a maior parte dos pedidos registrados nos dois anos: 393 em 2024 (52,5% do ano) e 410 em 2025 (54,8%). Novembro, depois do Halloween, ainda teve 174 e 182 pedidos, mais que agosto e setembro (no máximo 104). Uma possível explicação são promoções pós-Halloween ou festas realizadas depois da data, mas os dados não informam o motivo. Limitação: as contagens incluem pedidos cancelados, cobrem apenas agosto a novembro e a base é fictícia. Como sugestão de negócio, a empresa poderia investigar outras datas temáticas, mas esta base não tem dados para avaliar essa ideia.